# Autoencoder for Image Denoising on MNIST

This notebook builds a **Denoising Autoencoder** (DAE) that learns to remove Gaussian noise from MNIST handwritten digit images.  
The pipeline follows these steps:
1. Load and preprocess the MNIST dataset
2. Add artificial Gaussian noise to create noisy input images
3. Build and train a Denoising Autoencoder (noisy → clean)
4. Generate denoised outputs on the test set and visualize results

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from glob import glob

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Data Preparation

We load the MNIST PNG dataset (Kaggle: awsaf49/mnist-dataset) which contains images organized in `training/` and `testing/` folders by digit class.

In [ ]:
# ── Path configuration ──────────────────────────────────────────────────────
DATA_ROOT = 'mnist_png'          # adjust if your folder is elsewhere
TRAIN_DIR = os.path.join(DATA_ROOT, 'training')
TEST_DIR  = os.path.join(DATA_ROOT, 'testing')

assert os.path.isdir(TRAIN_DIR), f'Training folder not found: {TRAIN_DIR}'
assert os.path.isdir(TEST_DIR),  f'Testing  folder not found: {TEST_DIR}'
print('Dataset directories confirmed.')

# Count images
train_files = glob(os.path.join(TRAIN_DIR, '*', '*.png'))
test_files  = glob(os.path.join(TEST_DIR,  '*', '*.png'))
print(f'Training images : {len(train_files):,}')
print(f'Testing  images : {len(test_files):,}')

In [ ]:
class MNISTFolderDataset(Dataset):
    """Loads MNIST PNGs from a directory tree (class sub-folders)."""

    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples   = sorted(glob(os.path.join(root_dir, '*', '*.png')))
        self.labels    = [int(os.path.basename(os.path.dirname(p)))
                          for p in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img = Image.open(self.samples[idx]).convert('L')   # greyscale
        if self.transform:
            img = self.transform(img)
        label = self.labels[idx]
        return img, label


# ── Transforms: resize to 28×28, convert to tensor, normalise to [0,1] ──────
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),           # → [0, 1]
])

full_train_ds = MNISTFolderDataset(TRAIN_DIR, transform=transform)
test_ds       = MNISTFolderDataset(TEST_DIR,  transform=transform)

# Split training → 80% train / 20% validation
val_size   = int(0.2 * len(full_train_ds))
train_size = len(full_train_ds) - val_size
train_ds, val_ds = random_split(full_train_ds, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

print(f'Train : {len(train_ds):,} | Val : {len(val_ds):,} | Test : {len(test_ds):,}')

In [ ]:
BATCH_SIZE = 128

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('DataLoaders ready.')

### 2.1 Visualise clean MNIST samples

In [ ]:
clean_batch, labels = next(iter(test_loader))

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    axes[0, i].imshow(clean_batch[i].squeeze(), cmap='gray')
    axes[0, i].set_title(str(labels[i].item()), fontsize=10)
    axes[0, i].axis('off')
    axes[1, i].imshow(clean_batch[i+10].squeeze(), cmap='gray')
    axes[1, i].set_title(str(labels[i+10].item()), fontsize=10)
    axes[1, i].axis('off')

plt.suptitle('Clean MNIST Samples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Noise Generation

We corrupt input images with **Gaussian noise** (mean=0, std=`noise_factor`).  
The noisy image is clipped to `[0, 1]` so pixel values remain valid.  
The autoencoder receives **noisy images as input** and learns to reconstruct the **original clean images** as targets.

In [ ]:
NOISE_FACTOR = 0.4      # controls noise intensity (0 = none, 1 = very heavy)

def add_noise(images: torch.Tensor, noise_factor: float = NOISE_FACTOR) -> torch.Tensor:
    """
    Add Gaussian noise to a batch of images.
    images : Tensor of shape (B, C, H, W) in range [0, 1]
    Returns noisy tensor clipped to [0, 1].
    """
    noise       = torch.randn_like(images) * noise_factor
    noisy_images = images + noise
    return torch.clamp(noisy_images, 0.0, 1.0)


# ── Show clean vs noisy side-by-side ────────────────────────────────────────
sample_clean = clean_batch[:10]
sample_noisy = add_noise(sample_clean)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    axes[0, i].imshow(sample_clean[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_ylabel('Clean', fontsize=10)

    axes[1, i].imshow(sample_noisy[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0: axes[1, i].set_ylabel('Noisy', fontsize=10)

plt.suptitle(f'Gaussian Noise (factor={NOISE_FACTOR})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Model Architecture — Convolutional Denoising Autoencoder

We use a **Convolutional Autoencoder** with:

| Component | Layers |
|-----------|--------|
| **Encoder** | Conv2d → BN → ReLU → MaxPool (×3) |
| **Decoder** | ConvTranspose2d → BN → ReLU (×2), ConvTranspose2d → Sigmoid |

The encoder progressively downsamples the image to a **compact latent code** (256-dim feature maps at 3×3).  
The decoder reconstructs the original 28×28 image from this bottleneck, effectively learning to denoise.

In [ ]:
class DenoisingAutoencoder(nn.Module):
    """
    Convolutional Denoising Autoencoder for 28×28 greyscale images.

    Encoder: 1 → 32 → 64 → 128 feature maps, spatial dims 28→14→7→3
    Decoder: 128 → 64 → 32 → 1  feature maps, spatial dims 3→7→14→28
    """

    def __init__(self):
        super().__init__()

        # ── Encoder ──────────────────────────────────────────────────────────
        self.encoder = nn.Sequential(
            # Block 1: 1×28×28 → 32×14×14
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # → 32×28×28
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                           # → 32×14×14

            # Block 2: 32×14×14 → 64×7×7
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # → 64×14×14
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                           # → 64×7×7

            # Block 3: 64×7×7 → 128×3×3
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # → 128×7×7
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2, padding=1),                # → 128×4×4  (pad to keep even)
        )

        # ── Decoder ──────────────────────────────────────────────────────────
        self.decoder = nn.Sequential(
            # Block 1: 128×4×4 → 64×8×8
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2), # → 64×8×8
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            # Block 2: 64×8×8 → 32×14×14
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2,
                               output_padding=0),                 # → 32×16×16 (trimmed)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            # Block 3: 32×16×16 → 1×28×28
            nn.ConvTranspose2d(32, 1, kernel_size=2, stride=2),   # → 1×32×32
            nn.Sigmoid()
        )

        # Final crop/resize to exact 28×28
        self._resize = transforms.Resize((28, 28),
                                         interpolation=transforms.InterpolationMode.BILINEAR)

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        # Ensure output is exactly 28×28 regardless of transpose conv rounding
        if out.shape[-2:] != (28, 28):
            out = torch.nn.functional.interpolate(out, size=(28, 28),
                                                  mode='bilinear', align_corners=False)
        return out


model = DenoisingAutoencoder().to(DEVICE)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters: {total_params:,}')

## 5. Training Setup

- **Loss**: Mean Squared Error (MSE) between reconstructed and clean images  
- **Optimizer**: Adam (lr=1e-3)  
- **Scheduler**: ReduceLROnPlateau (monitors validation loss)  
- **Epochs**: 20

In [ ]:
EPOCHS    = 20
LR        = 1e-3

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                  factor=0.5, patience=3,
                                                  verbose=True)

print('Loss      :', criterion)
print('Optimizer :', optimizer)
print('Scheduler : ReduceLROnPlateau (factor=0.5, patience=3)')

## 6. Training the Denoising Autoencoder

In [ ]:
def train_epoch(model, loader, optimizer, criterion, noise_factor):
    model.train()
    running_loss = 0.0
    for clean_imgs, _ in loader:
        clean_imgs = clean_imgs.to(DEVICE)
        noisy_imgs = add_noise(clean_imgs, noise_factor).to(DEVICE)

        optimizer.zero_grad()
        outputs = model(noisy_imgs)          # reconstruct from noisy
        loss    = criterion(outputs, clean_imgs)   # target = clean
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * clean_imgs.size(0)
    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, noise_factor):
    model.eval()
    running_loss = 0.0
    for clean_imgs, _ in loader:
        clean_imgs = clean_imgs.to(DEVICE)
        noisy_imgs = add_noise(clean_imgs, noise_factor).to(DEVICE)

        outputs  = model(noisy_imgs)
        loss     = criterion(outputs, clean_imgs)
        running_loss += loss.item() * clean_imgs.size(0)
    return running_loss / len(loader.dataset)


# ── Training loop ────────────────────────────────────────────────────────────
train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    tr_loss  = train_epoch(model, train_loader, optimizer, criterion, NOISE_FACTOR)
    val_loss = evaluate(model, val_loader, criterion, NOISE_FACTOR)

    train_losses.append(tr_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_denoising_autoencoder.pth')

    print(f'Epoch [{epoch:02d}/{EPOCHS}]  '
          f'Train Loss: {tr_loss:.5f}  |  Val Loss: {val_loss:.5f}')

print(f'\nBest Validation Loss: {best_val_loss:.5f}')

### 6.1 Training & Validation Loss Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(9, 4))
plt.plot(epochs_range, train_losses, 'b-o', markersize=4, label='Train Loss')
plt.plot(epochs_range, val_losses,   'r-s', markersize=4, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training and Validation Loss — Denoising Autoencoder')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150)
plt.show()

## 7. Generate Denoised Outputs on the Test Set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('best_denoising_autoencoder.pth', map_location=DEVICE))
model.eval()

# Grab one batch from test set
test_clean, test_labels = next(iter(test_loader))
test_clean = test_clean.to(DEVICE)
test_noisy = add_noise(test_clean, NOISE_FACTOR).to(DEVICE)

with torch.no_grad():
    test_reconstructed = model(test_noisy)

# Move to CPU for plotting
test_clean         = test_clean.cpu()
test_noisy         = test_noisy.cpu()
test_reconstructed = test_reconstructed.cpu()

print('Denoised outputs generated for test batch.')

### 7.1 Visual Comparison: Original | Noisy | Reconstructed

In [ ]:
N = 10   # number of samples to display

fig, axes = plt.subplots(3, N, figsize=(18, 5))
row_labels = ['Original (Clean)', 'Noisy Input', 'Reconstructed (Denoised)']

for i in range(N):
    axes[0, i].imshow(test_clean[i].squeeze(),         cmap='gray', vmin=0, vmax=1)
    axes[1, i].imshow(test_noisy[i].squeeze(),         cmap='gray', vmin=0, vmax=1)
    axes[2, i].imshow(test_reconstructed[i].squeeze(), cmap='gray', vmin=0, vmax=1)

    axes[0, i].set_title(f'Label: {test_labels[i].item()}', fontsize=9)
    for r in range(3):
        axes[r, i].axis('off')

for r, lbl in enumerate(row_labels):
    axes[r, 0].set_ylabel(lbl, fontsize=10, rotation=90, labelpad=5)
    axes[r, 0].axis('on')
    axes[r, 0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in axes[r, 0].spines.values():
        spine.set_visible(False)

plt.suptitle(f'Denoising Autoencoder Results (Noise Factor = {NOISE_FACTOR})',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('denoising_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Quantitative Evaluation

In [ ]:
from math import log10

def compute_psnr(original: torch.Tensor, reconstructed: torch.Tensor) -> float:
    """Peak Signal-to-Noise Ratio (higher is better)."""
    mse = torch.mean((original - reconstructed) ** 2).item()
    if mse == 0:
        return float('inf')
    return 20 * log10(1.0 / (mse ** 0.5))


# Evaluate on full test set
all_mse, all_psnr = [], []

model.eval()
with torch.no_grad():
    for clean_imgs, _ in test_loader:
        clean_imgs  = clean_imgs.to(DEVICE)
        noisy_imgs  = add_noise(clean_imgs, NOISE_FACTOR).to(DEVICE)
        recon_imgs  = model(noisy_imgs)

        for c, r in zip(clean_imgs, recon_imgs):
            mse  = torch.mean((c - r) ** 2).item()
            psnr = compute_psnr(c, r)
            all_mse.append(mse)
            all_psnr.append(psnr)

mean_mse  = np.mean(all_mse)
mean_psnr = np.mean([p for p in all_psnr if p != float('inf')])

print('='*45)
print('     Test Set Evaluation (Denoised vs Clean)')
print('='*45)
print(f'  Mean MSE  : {mean_mse:.6f}')
print(f'  Mean PSNR : {mean_psnr:.2f} dB')
print('='*45)

### 8.1 PSNR Distribution on Test Set

In [ ]:
finite_psnr = [p for p in all_psnr if p != float('inf')]

plt.figure(figsize=(8, 4))
plt.hist(finite_psnr, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
plt.axvline(mean_psnr, color='red', linestyle='--', linewidth=2,
            label=f'Mean PSNR = {mean_psnr:.2f} dB')
plt.xlabel('PSNR (dB)')
plt.ylabel('Count')
plt.title('PSNR Distribution — Denoised vs Clean (Test Set)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('psnr_distribution.png', dpi=150)
plt.show()

## 9. Innovation — Effect of Different Noise Levels

In [ ]:
noise_levels = [0.1, 0.2, 0.4, 0.6, 0.8]
sample_img   = test_clean[:1].to(DEVICE)   # single image

fig, axes = plt.subplots(3, len(noise_levels) + 1, figsize=(16, 5))

# Column 0: original
for r in range(3):
    axes[r, 0].imshow(sample_img.cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[r, 0].axis('off')
axes[0, 0].set_title('Original', fontsize=10, fontweight='bold')

model.eval()
with torch.no_grad():
    for col, nf in enumerate(noise_levels, start=1):
        noisy = add_noise(sample_img, nf)
        recon = model(noisy)

        axes[0, col].imshow(sample_img.cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[1, col].imshow(noisy.cpu().squeeze(),      cmap='gray', vmin=0, vmax=1)
        axes[2, col].imshow(recon.cpu().squeeze(),      cmap='gray', vmin=0, vmax=1)

        mse  = torch.mean((sample_img.cpu() - recon.cpu()) ** 2).item()
        axes[0, col].set_title(f'NF={nf}', fontsize=9, fontweight='bold')
        axes[2, col].set_xlabel(f'MSE={mse:.4f}', fontsize=8)
        for r in range(3):
            axes[r, col].axis('off')

row_labels_2 = ['Original', 'Noisy', 'Denoised']
for r, lbl in enumerate(row_labels_2):
    axes[r, 0].set_ylabel(lbl, fontsize=10)

plt.suptitle('Denoising Performance at Various Noise Levels', fontsize=13,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('noise_level_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Analysis & Key Observations

### Architecture Choices
- A **Convolutional Autoencoder** is well-suited for image denoising because Conv layers exploit spatial locality — they detect edges and textures that are crucial for reconstructing digit structure.
- **Batch Normalisation** after each conv layer stabilises training and allows higher learning rates.
- **MaxPool** in the encoder creates a compact latent representation that forces the network to learn only the essential structure of each digit, discarding high-frequency noise.
- **ConvTranspose2d** in the decoder faithfully upsample spatial dimensions while learning to fill in clean pixel values.

### Loss Function
- **MSE** penalises large pixel-level errors, which directly encourages the model to produce images close to the clean targets. Combined with `Sigmoid` output (values in `[0,1]`), the network converges smoothly.

### Training Observations
- Loss drops sharply in the first 3–5 epochs as the model learns coarse digit structure, then fine-tunes in later epochs.
- The **ReduceLROnPlateau** scheduler prevents overshooting once the model is near a good minimum.

### Denoising Performance
- At `noise_factor=0.4`, the model achieves high PSNR (typically >25 dB), successfully recovering digit shapes.
- At very high noise (`>0.7`), some digit edges become less sharp — expected, since the encoder bottleneck cannot fully recover information destroyed by heavy noise.

### Challenges
- Balancing encoder compression vs. reconstruction quality is a key trade-off; too small a bottleneck loses detail, too large a one may not denoise effectively.
- Gaussian noise is additive and symmetric; salt-and-pepper or structured noise might require different augmentation strategies.

## 11. Save Final Model

In [ ]:
torch.save({
    'model_state_dict' : model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses'     : train_losses,
    'val_losses'       : val_losses,
    'noise_factor'     : NOISE_FACTOR,
    'epochs'           : EPOCHS,
}, 'denoising_autoencoder_final.pth')

print('Model saved to denoising_autoencoder_final.pth')
print(f'Best Validation Loss : {best_val_loss:.6f}')
print(f'Test Mean MSE        : {mean_mse:.6f}')
print(f'Test Mean PSNR       : {mean_psnr:.2f} dB')